Community Detection using Binary Encoding


In [ ]:
!jupyter nbconvert --to script communityGraphFunctions.ipynb quantumUtility.ipynb

In [ ]:
from communityGraphFunctions import *
from quantumUtility import *
import math

In [ ]:
# Define global parameters for the problem
N = 6  # Number of nodes in the graph
K = 2  # Number of communities
P = 1  # Number of QAOA layers
IBM_KEY = '' # IBM Quantum API key for authenticating with Qiskit Runtime services
# Initial Parameters for optimization
INITIAL_GAMMA = np.pi # Initial value for the gamma parameter in QAOA, often chosen as pi
INITIAL_BETA = np.pi/2 # Initial value for the beta parameter in QAOA, often chosen as pi/2

**Cost function for Binary encoded**

$H = -\frac{1}{4m}\sum_{i,j} B_{ij} \sum_{b=1}^{\text{num\_bits}} Z_{i b} Z_{j b}$

In [ ]:
def create_cost_hamiltonian(G, modularity_matrix, N, num_bits):
    """Construct the cost Hamiltonian for the binary‑encoded community
    detection problem.

    The Hamiltonian has each original node `i` is represented by `num_bits` qubits
    (the b-th qubit of node i has index `i*num_bits + b`).

    Args:
        G (networkx.Graph): The original graph for which the modularity
                            matrix and Cost Hamiltonian are derived.
        modularity_matrix (np.ndarray):  NxN matrix B_{ij}.
        N (int):                        number of nodes in the graph.
        num_bits (int):                 number of binary bits per node.

    Returns:
        tuple: (num_qubits, cost_hamiltonian) where
               num_qubits = N * num_bits and the second element is a
               `SparsePauliOp` encoding the above Hamiltonian.
    """
    num_qubits = N * num_bits
    m = G.number_of_edges()
    pauli_terms = []

    # loop over unordered pairs of nodes
    for i in range(N):
        for j in range(i + 1, N):
            coeff = -(1 / (4 * m)) * modularity_matrix[i, j]
            if coeff == 0:
                continue
            # sum over bits: one ZZ term per bit b (not a product over all bits)
            for b in range(num_bits):
                pauli_str = ['I'] * num_qubits
                pauli_str[i * num_bits + b] = 'Z'
                pauli_str[j * num_bits + b] = 'Z'
                pauli_terms.append(("".join(reversed(pauli_str)), coeff))

    cost_hamiltonian = SparsePauliOp.from_list(pauli_terms,
                                               num_qubits=num_qubits)
    return num_qubits, cost_hamiltonian

In [ ]:
def group_nodes_by_community(G, final_answer, K):
    """Decode the binary‑encoded bitstring produced by the QAOA run and
    colour the nodes of `G` according to the community they belong to.

    Parameters
    ----------
    G : networkx.Graph
        The original graph whose nodes are numbered `0 … N‑1`.
    final_answer : list of str
        A list of `'0'`/`'1'` characters of length `N*num_bits` returned by
        `sampleOutput` (see cell 6).  The bits for node `i` occupy
        `final_answer[i*num_bits:(i+1)*num_bits]`.
    K : int
        The number of communities that was requested; used to clamp the
        decoded integer to a valid range.

    The function sets a node attribute `"community"` on `G` and then calls
    `display_graph` to show the coloured graph.
    """
    N = G.number_of_nodes()
    if N == 0:
        return

    num_bits = len(final_answer) // N
    assignment = {}
    for i in range(N):
        bits = final_answer[i * num_bits : (i + 1) * num_bits]
        value = int("".join(bits), 2)
        if value >= K:
            # wrap/clip the value into the requested community range
            value = value % K
        assignment[i] = value

    nx.set_node_attributes(G, assignment, name="community")

    # generate a colour list (tab10 has at least 10 distinct colours)
    cmap = plt.get_cmap("tab10")
    colors = [cmap(assignment[i] % cmap.N) for i in range(N)]

    display_graph(
        G,
        title="Detected communities (binary encoding)",
        node_colors=colors
    )

In [ ]:
def binary_quantum_optimization(G, N, K, P, LAMBDA_PENALTY):
    """Runs the full QAOA pipeline for community detection using binary encoding.

    Constructs the supernode graph, cost Hamiltonian, and QAOA ansatz for the
    binary-encoded community detection problem. Runs classical optimisation on
    a local AerSimulator, then samples the optimised circuit and decodes the
    result into community assignments.

    Args:
        G (networkx.Graph): The problem graph.
        N (int): The number of nodes in the graph.
        K (int): The requested number of communities.
        P (int): The number of QAOA layers.
        LAMBDA_PENALTY (float): Penalty coefficient for the constraint Hamiltonian.
    """
    display_graph(G, title="Initial Graph for Community Detection") # Visualise the problem graph

    # The maximum number of communities cannot exceed the total number of nodes
    KMAX = G.number_of_nodes()
    if K > KMAX:
        K = KMAX # Clamp K to the number of nodes to keep the problem well-defined
    print(f"KMAX = {KMAX}")

    # Minimum number of bits needed to represent K communities as a binary integer
    num_bits = math.ceil(math.log2(K)) # Calculate num_bits based on K

    G_onehot_supernodes_labeled = create_supernode(G, num_bits) # Create the supernode graph with num_bits copies per original node
    display_graph(G_onehot_supernodes_labeled, "Supernode Structure") # Visualise the supernode structure

    modularity_matrix = create_modularity_matrix(G, N) # Calculate the modularity matrix for the graph

    num_qubits, cost_hamiltonian = create_cost_hamiltonian(G, modularity_matrix, N, num_bits) # Construct the Cost Hamiltonian for the binary-encoded community detection problem

    # QAOA ansatz construction
    circuit = create_qaoa_ansatz(cost_hamiltonian, P, N, K) # Create the QAOA quantum circuit ansatz
    print(f"\n\nQAOA Ansatz: ")
    draw_circuit_mpl(circuit, "QAOA Ansatz Circuit") # Draw and display the QAOA ansatz circuit

    # Optimisation process
    initial_params = [INITIAL_GAMMA, INITIAL_BETA] # Set initial parameters for the QAOA optimisation
    transpiled_qc, backend = run_ansatz_simulator_circuit_prep(circuit) # Prepare the circuit for simulation (transpile for AerSimulator)
    result, value_tracker = run_optimizer_simulator(transpiled_qc, backend, initial_params, cost_hamiltonian) # Run the classical optimiser to find optimal QAOA parameters
    plot_optimizer_results(value_tracker) # Plot the cost function's value over optimisation iterations

    # Result extraction and interpretation
    optimized_circuit = transpiled_qc.assign_parameters(result.x) # Assign the optimised parameters to the quantum circuit
    print(f"\n\nThe circuit with optimized parameters: ")
    draw_circuit_mpl(optimized_circuit, "Optimized Circuit") # Draw and display the optimised circuit

    # Sample measurement outcomes from the optimised circuit to get the solution bitstring
    final_answer = sample_output(optimized_circuit, backend, num_qubits)
    print(f"\n\nThe optimized bitstring for the problem in hand: ")
    final_answer = [str(i) for i in final_answer]
    print(final_answer)

    print(f"\n\nThe community division w.r.t. the input k = {K}:")
    group_nodes_by_community(G, final_answer, K) # Group nodes into communities based on the final bitstring and visualise the result


G, N = create_clique_chain_graph(3, 3)
K = 3  # Number of communities
P = 1
LAMBDA_PENALTY = 1 # Penalty parameter for the Hamiltonian to enforce constraints
binary_quantum_optimization(G, N, K, P, LAMBDA_PENALTY)

**Classical Implementation**

$ H_C = \max_{\mathbf{x}} \; \frac{1}{2m} \sum_{n} \sum_{i,j} B_{ij} \left( x_{i,n} - \frac{1}{2} \right) \left( x_{j,n} - \frac{1}{2} \right)$
\
\
OR
\
\
$ H_C = \max_{\mathbf{x}} \; \frac{1}{2m} \sum_{n} \mathbf{v}_n^T B \, \mathbf{v}_n, \quad \mathbf{v}_n = \mathbf{X}_{:,n} - \tfrac{1}{2}$

In [ ]:
'''
Objective (maximise):
  H_C = (1/2m) * Σ_n Σ_{i,j} B_ij * (x_{i,n} - 0.5) * (x_{j,n} - 0.5)

Variables: X[i, n] = x_{i,n}  — n-th bit of node i's community label
           continuous relaxation x_{i,n} ∈ [0, 1]
Decode   : round X to {0,1}, read each row as a binary integer → community index
Optimiser: scipy COBYLA (no gradients needed, same family as QAOA optimizer)
'''


import math
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy.optimize import minimize


def build_modularity_matrix_binary(G, N):
    """Constructs the modularity matrix B for the binary-encoded community detection problem.

    Computes B = A - (deg * deg^T) / (2m), where A is the adjacency matrix,
    deg is the degree vector, and m is the total number of edges.

    Args:
        G (networkx.Graph): The original graph for which the modularity matrix is derived.
        N (int): The number of nodes in the graph.

    Returns:
        tuple: A tuple containing:
            - B (numpy.ndarray): The NxN modularity matrix.
            - m (float): The total number of edges in the graph.
    """
    A = nx.to_numpy_array(G, nodelist=sorted(G.nodes())) # Convert graph to adjacency matrix
    m = A.sum() / 2 # Total number of edges (sum of all entries divided by 2)
    deg = A.sum(axis=1) # Degree vector: sum of each row gives the degree of each node
    B = A - np.outer(deg, deg) / (2 * m) # Modularity matrix: B_ij = A_ij - k_i*k_j / (2m)

    print("m =", m)
    print("NaN:", np.any(np.isnan(B)))
    print("Inf:", np.any(np.isinf(B)))
    print("B =", B)
    
    return B, m


def binary_objective(x_flat, B, N, num_bits, m, tracker):
    """Computes the binary-encoded modularity objective for the continuous relaxation.

    Evaluates H_C = (1/2m) * Σ_n v_n^T B v_n, where v_n = X[:, n] - 0.5.
    The value is negated so that minimisation routines can be used to
    maximise the true objective.

    Args:
        x_flat (numpy.ndarray): Flattened assignment matrix of shape (N*num_bits,).
        B (numpy.ndarray): The modularity matrix for the graph.
        N (int): The number of nodes in the graph.
        num_bits (int): The number of binary bits used to encode each community label.
        m (float): The total number of edges in the graph.
        tracker (list): Accumulates the objective value at each evaluation for convergence plots.

    Returns:
        float: Negated objective value (for use with minimisation routines).
    """
    X = x_flat.reshape(N, num_bits) # Reshape flat array into (N x num_bits) assignment matrix

    value = 0.0
    for bit in range(num_bits):
        v = X[:, bit] - 0.5 # Centre the bit values around zero (maps {0,1} → {-0.5, +0.5})
        value += v @ B @ v # Accumulate the quadratic modularity contribution for this bit
    value /= (2 * m) # Normalise by twice the number of edges

    tracker.append(value) # Record the current objective value for convergence tracking
    return -value # Negate to convert maximisation to minimisation


def run_binary_classical(B, N, K, m, n_restarts=10, seed=42):
    """Optimises the binary objective using COBYLA with multiple random restarts.

    Uses a continuous relaxation of the binary community assignment problem,
    with box constraints keeping all variables in [0, 1]. Multiple random
    restarts reduce the risk of converging to a poor local optimum.

    Args:
        B (numpy.ndarray): The modularity matrix for the graph.
        N (int): The number of nodes in the graph.
        K (int): The number of communities.
        m (float): The total number of edges in the graph.
        n_restarts (int): Number of independent random restarts.
        seed (int): Random seed for reproducibility.

    Returns:
        tuple: A tuple containing:
            - best_result (scipy.optimize.OptimizeResult): The best optimisation result found.
            - tracker (list[float]): Objective values recorded across all evaluations.
            - num_bits (int): The number of bits used to encode each community label.
    """
    num_bits = math.ceil(math.log2(max(K, 2))) # Minimum bits needed to represent K communities
    rng = np.random.default_rng(seed)
    best_result = None
    tracker = []

    # Box constraints: 0 <= x_{i,n} <= 1 for all i, n
    constraints = []
    for idx in range(N * num_bits):
        constraints.append({"type": "ineq", "fun": lambda x, i=idx:  x[i]})       # x_{i,n} >= 0
        constraints.append({"type": "ineq", "fun": lambda x, i=idx:  1 - x[i]})   # x_{i,n} <= 1

    for _ in range(n_restarts):
        x0 = rng.uniform(0, 1, size=N * num_bits) # Random initialisation within the feasible box

        result = minimize(
            binary_objective,
            x0,
            args=(B, N, num_bits, m, tracker),
            method="COBYLA",
            constraints=constraints,
            tol=1e-6,
            options={"maxiter": 5000},
        )

        if best_result is None or result.fun < best_result.fun:
            best_result = result # Keep the result with the lowest (negated) objective value

    return best_result, tracker, num_bits


def decode_binary_assignments(x_flat, N, num_bits, K):
    """Decodes the continuous relaxation solution into integer community assignments.

    Rounds each continuous bit value to the nearest integer (0 or 1), then
    reads each node's bit row as a big-endian binary integer, wrapping the
    result into the valid community range [0, K).

    Args:
        x_flat (numpy.ndarray): Flattened assignment matrix of shape (N*num_bits,).
        N (int): The number of nodes in the graph.
        num_bits (int): The number of binary bits used to encode each community label.
        K (int): The number of communities.

    Returns:
        numpy.ndarray: Integer array of shape (N,) giving the community index for each node.
    """
    X = x_flat.reshape(N, num_bits)
    X_binary = (X >= 0.5).astype(int) # Round each continuous value to the nearest bit

    assignments = []
    for i in range(N):
        community = int("".join(str(b) for b in X_binary[i]), 2) % K # Read row as big-endian binary integer and wrap into [0, K)
        assignments.append(community)
    return np.array(assignments)


def plot_binary_convergence(tracker):
    """Plots the objective value over function evaluations during COBYLA optimisation.

    Args:
        tracker (list[float]): Sequence of objective values recorded during optimisation.
    """
    plt.figure(figsize=(10, 4))
    plt.plot(tracker)
    plt.xlabel("Function evaluation")
    plt.ylabel("H_C")
    plt.title("Classical COBYLA — binary encoding convergence")
    plt.tight_layout()
    plt.show()


def plot_binary_communities(G, assignments, K):
    """Visualises the detected community structure of the graph using colour-coded nodes.

    Args:
        G (networkx.Graph): The original problem graph.
        assignments (numpy.ndarray): Integer array of community indices for each node.
        K (int): The number of communities.
    """
    cmap = plt.get_cmap("tab10")
    node_colors = [cmap(c % cmap.N) for c in assignments] # Assign a colour to each node based on its community
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(8, 6))
    nx.draw(G, pos=pos, node_color=node_colors, with_labels=True,
            node_size=700, font_color="white")
    plt.title(f"Detected Communities — binary encoding (K={K})")
    plt.show()


def binary_classical_optimization(G, N, K, lambda_penalty):
    """Runs the full classical COBYLA optimisation pipeline for the binary-encoded community detection problem.

    Constructs the modularity matrix, runs the continuous-relaxation COBYLA
    optimiser, decodes the result into community assignments, and displays
    the convergence plot and the final community partition.

    Args:
        G (networkx.Graph): The problem graph (unused; recreated internally via create_initial_graph).
        N (int): The number of nodes in the graph.
        K (int): The number of communities.
        lambda_penalty (float): Penalty coefficient (reserved for future constraint terms).
    """
    display_graph(G, title="Initial Graph for Community Detection")
    B, m = build_modularity_matrix_binary(G, N) # Construct the modularity matrix and retrieve the edge count

    result, tracker, num_bits_used = run_binary_classical(B, N, K, m) # Run COBYLA optimisation with multiple random restarts
    assignments = decode_binary_assignments(result.x, N, num_bits_used, K) # Decode continuous solution into hard community assignments

    print(f"num_bits used       : {num_bits_used}")
    print(f"Optimal H_C         : {-result.fun:.4f}")
    print(f"Community assignments: {assignments}")
    print(f"Function evaluations : {result.nfev}")

    plot_binary_convergence(tracker) # Plot the objective value over optimisation evaluations
    plot_binary_communities(G, assignments, K) # Visualise the final community partition on the graph


G, N = create_clique_chain_graph(3, 2)
K = 2  # Number of communities
LAMBDA_PENALTY = 1 # Penalty parameter for the Hamiltonian to enforce constraints

binary_classical_optimization(G, N, K, LAMBDA_PENALTY)